# **Evaluación corta — Regresión múltiple con datos de próstata**

El objetivo es estudiar variables asociadas al logaritmo del antígeno prostático específico (lpsa).



In [2]:
suppressPackageStartupMessages(
  library(dplyr)
)

In [3]:
#Cargamos la base de datos
df_prostate = read.table("/content/prostate.txt",header = TRUE)
#Visualizamos las primeras 6 filas
head(df_prostate)

,lcavol,lweight,age,lbph,svi,lcp,gleason,pgg45,lpsa,train
,<dbl>,<dbl>,<int>,<dbl>,<int>,<dbl>,<int>,<int>,<dbl>,<lgl>
1,-0.5798185,2.769459,50,-1.386294,0,-1.386294,6,0,-0.4307829,TRUE
2,-0.9942523,3.319626,58,-1.386294,0,-1.386294,6,0,-0.1625189,TRUE
3,-0.5108256,2.691243,74,-1.386294,0,-1.386294,7,20,-0.1625189,TRUE
4,-1.2039728,3.282789,58,-1.386294,0,-1.386294,6,0,-0.1625189,TRUE
5,0.7514161,3.432373,62,-1.386294,0,-1.386294,6,0,0.3715636,TRUE
6,-1.0498221,3.228826,50,-1.386294,0,-1.386294,6,0,0.7654678,TRUE


 1. (1 punto) **Inspección de la base de datos.** La función str nos permite ver la estructura de la base de datos.

*   **Pregunta:** A partir de str(df_prostate), indique cuántas observaciones y variables tiene la base.
*   **Respuesta:** hay 97 observaciones y 10 variables.




In [4]:
#La fun
str(df_prostate)

'data.frame':	97 obs. of  10 variables:
 $ lcavol : num  -0.58 -0.994 -0.511 -1.204 0.751 ...
 $ lweight: num  2.77 3.32 2.69 3.28 3.43 ...
 $ age    : int  50 58 74 58 62 50 64 58 47 63 ...
 $ lbph   : num  -1.39 -1.39 -1.39 -1.39 -1.39 ...
 $ svi    : int  0 0 0 0 0 0 0 0 0 0 ...
 $ lcp    : num  -1.39 -1.39 -1.39 -1.39 -1.39 ...
 $ gleason: int  6 6 7 6 6 6 6 6 6 6 ...
 $ pgg45  : int  0 0 20 0 0 0 0 0 0 0 ...
 $ lpsa   : num  -0.431 -0.163 -0.163 -0.163 0.372 ...
 $ train  : logi  TRUE TRUE TRUE TRUE TRUE TRUE ...


2. (1 punto) **Intepretación de código.**

*   **Pregunta:** Explique de manera breve, en sus propias palabras qué esta haciendo el siguiente código.
*   **Respuesta:** está separando manualmente la data entre set de entrenamiento y testeo, usando la columna "train" como criterio. también se imprime la dimensión de cada frame nuevo.



In [5]:

df_train <- df_prostate |> filter(train==TRUE)
print(dim(df_train))

df_test <- df_prostate |> filter(train==FALSE)
print(dim(df_test))

[1] 67 10
[1] 30 10


3. (1 punto) **Construcción del modelo** Considere inicialmente las variables `lcavol, lweight, age, svi`.


*   Ajuste un modelo lineal para explicar lpsa a partir de estas cuatro variables. Utilice `df_train`.
*   Escriba la ecuación del modelo estimado usando **los coeficientes que obtuvo en R.**
* De las cuatro variables, indique cuál presenta mayor evidencia de asociación con lpsa.

Incluya código utilizado y el resultado numérico que justifíca su respuesta.

**Respuesta:** una variable con asociacion a lpsa es lcavol, con un t valor de 5.48374 (grande) y p valor de 8.132306e-07 (muy pequeño).
**Ecuación:** lpsa=-0.64+0.53lcavol+0.77lweight-0.008age+0.53svi

In [6]:
#Matriz de diseño
n=length(df_train$lpsa)
one=rep(1,n)

#Crear X e y
X=cbind(one,df_train$lcavol,df_train$lweight,df_train$age,df_train$svi)
y=df_train$lpsa

#Estimador de minimos cuadrados
#(X^T X)^{-1} %*% X^Ty
beta=solve(t(X)%*%X)%*%t(X)%*%y
beta

one,-0.642189293
,0.531733914
,0.768120079
,-0.007839715
,0.528909376


In [9]:
# ajuste modelo
p=ncol(X) #parametros
df_r=n-p #df
r=y-X%*%beta #residuos (dato real - predicciones)
rss= t(r)%*%r #RSS (residuos ^2)
var=as.numeric(rss/(df_r)) #varianza
cat("varianza", var)
covar_matrix=var*solve(t(X)%*%X) #matriz de covarianzas y varianzas
covar_matrix
se=sqrt(diag(covar_matrix)) #standard error
head(df_prostate,0) #para ver los nombres de las columnas en orden (falta el intercepto)
cat("\n", "standard error", se)
t=beta/se #estadistico t
cat("\n", "t value", t)
pvalue=2*pt(-abs(t),df=df_r) #p valor
cat("\n","p-value",pvalue)

varianza 0.5598731

,one,,,,
one,0.928182365,0.0212686861,-0.1085155631,-0.0085395336,-0.0061203884
,0.021268686,0.0094023194,-0.0036297546,-0.0002635934,-0.0151419186
,-0.108515563,-0.0036297546,0.0438540766,-0.0007028529,-0.0010241378
,-0.008539534,-0.0002635934,-0.0007028529,0.0001759052,0.0002018007
,-0.006120388,-0.0151419186,-0.0010241378,0.0002018007,0.0744011095


lcavol,lweight,age,lbph,svi,lcp,gleason,pgg45,lpsa,train
<dbl>,<dbl>,<int>,<dbl>,<int>,<dbl>,<int>,<int>,<dbl>,<lgl>



 standard error 0.9634222 0.09696556 0.2094136 0.01326292 0.2727657
 t value -0.666571 5.48374 3.667956 -0.5911 1.939061
 p-value 0.5075192 8.132306e-07 0.0005097004 0.5566023 0.05704827

4. (1 punto) **No basta con mirar el p-valor**:
Seleccione una variable (la que usted quiera) que sea significativa a nivel α=0.1 en su modelo. Explique en **dos lineas** qué significa su coeficiente estimado en el contexto del problema. Luego responda: ¿sería correcto decir que esta variable causa un aumento del PSA? Justifique su respuesta.

**Respuesta:** si elijo la variable "svi", su coeficiente significa que, manteniendo fijos lcavol, lweight y age, los pacientes con invasión vesícula seminal (svi=1) presentan, en promedio, un lpsa 0.529 unidades más alto que los pacientes sin invasión (svi=0).
A pesar de lo anterior, no sería correcto decir que esta variable (svi) causa un aumento del psa, esto significaría causalidad, y el modelo no captura esta relación.

In [8]:
modelo1 <- lm(lpsa ~lcavol + lweight + age + svi, data = df_train)
summary(modelo1)


Call:
lm(formula = lpsa ~ lcavol + lweight + age + svi, data = df_train)

Residuals:
     Min       1Q   Median       3Q      Max 
-1.69266 -0.48898 -0.00507  0.49826  1.69232 

Coefficients:
            Estimate Std. Error t value Pr(>|t|)    
(Intercept) -0.64219    0.96342  -0.667  0.50752    
lcavol       0.53173    0.09697   5.484 8.13e-07 ***
lweight      0.76812    0.20941   3.668  0.00051 ***
age         -0.00784    0.01326  -0.591  0.55660    
svi          0.52891    0.27277   1.939  0.05705 .  
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

Residual standard error: 0.7482 on 62 degrees of freedom
Multiple R-squared:  0.6395,	Adjusted R-squared:  0.6162 
F-statistic: 27.49 on 4 and 62 DF,  p-value: 3.834e-13


5. (2 puntos) **¿Conviene complicar el modelo?** Ajuste ahora:

In [11]:
modelo2 <- lm(lpsa ~ lcavol + lweight + age + svi +lbph + lcp + gleason + pgg45, data = df_train)
summary(modelo2)



Call:
lm(formula = lpsa ~ lcavol + lweight + age + svi + lbph + lcp + 
    gleason + pgg45, data = df_train)

Residuals:
     Min       1Q   Median       3Q      Max 
-1.64870 -0.34147 -0.05424  0.44941  1.48675 

Coefficients:
             Estimate Std. Error t value Pr(>|t|)    
(Intercept)  0.429170   1.553588   0.276  0.78334    
lcavol       0.576543   0.107438   5.366 1.47e-06 ***
lweight      0.614020   0.223216   2.751  0.00792 ** 
age         -0.019001   0.013612  -1.396  0.16806    
svi          0.737209   0.298555   2.469  0.01651 *  
lbph         0.144848   0.070457   2.056  0.04431 *  
lcp         -0.206324   0.110516  -1.867  0.06697 .  
gleason     -0.029503   0.201136  -0.147  0.88389    
pgg45        0.009465   0.005447   1.738  0.08755 .  
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

Residual standard error: 0.7123 on 58 degrees of freedom
Multiple R-squared:  0.6944,	Adjusted R-squared:  0.6522 
F-statistic: 16.47 on 8 and 58 DF,  p-value: 2.0

Compare este modelo con el modelo anterior usadno:
* R^2 ajustado **modelo2**
* AIC **modelo2**

Si tuviera que escoger solo uno de los dos modelos ¿Cuál escogería?
**por ambos criterios se elije el modelo 2**

In [22]:
#Comparacion R^2
ymean=mean(y)
sct1=sum((y-ymean)^2)
sce1= rss
scr1=sum(sct1-sce1)
r2adj_modelo1=1-(sce1/df_r)/(sct1/(n-1))

cat("R^2adj modelo 1:", r2adj_modelo1,"\n","R^2adj modelo 2:","0.6522")

R^2adj modelo 1: 0.6162124 
 R^2adj modelo 2: 0.6522

In [25]:
#Comparacion AIC
AIC(modelo1,modelo2)


,df,AIC
,<dbl>,<dbl>
modelo1,6,158.0783
modelo2,10,155.0101


In [29]:
logL=-n/2*log(2*pi)-n/2*log(rss/n)-n/2 #preguntar
k=p+1

AIC1=-2*logL+2*k
AIC1

158.0783


Considere el siguiente código, que obtiene el error predictivo asociado al segundo modelo.

In [31]:
pred_2 <- predict(modelo2, newdata = df_test)
mean((df_test$lpsa - pred_2)^2)

[1] 0.521274

Replique el código anterior para obtener el error predictivo asociado al primer modelo que ajusto. ¿El modelo que prefirió usando AIC y R^2 ajustado es también el que presenta menor error de predicción en df_test? Comente brevemente.

**Respuesta:** no, el modelo preferido por el AIC y R^2adj no es el que tiene menor error de prediccion en df_test (modelo 1 tiene menor mse). Esto ocurre porque AIC y R^2 se calculan sobre el mismo df_train (evaluamos sobre los datos con que se entrena el modelo), en cambio al predecir, estamos evaluando sobre datos nuevos, evaluamos la capacidad predictiva del modelo.

In [32]:
pred_1 <- predict(modelo1, newdata = df_test)
mean((df_test$lpsa - pred_1)^2)

[1] 0.3878768